# BetaEarth: AlphaEarth Embedding Emulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/betaearth)](https://pypi.org/project/betaearth/)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97-Models-yellow)](https://huggingface.co/asterisk-labs)

This notebook shows how to predict dense 10m geospatial embeddings from Sentinel-2 imagery using BetaEarth.

**What you'll do:**
1. Install BetaEarth and load the best model
2. Load a real Sentinel-2 tile from Major TOM (single row, no bulk download)
3. Predict embeddings and visualise as PCA-RGB
4. Compare model variants (full, no-FiLM, RGB-only)

## 1. Install

In [ ]:
!pip install -q betaearth rasterio matplotlib scikit-learn fsspec pyarrow

## 2. Load model

The default model is the **curriculum (robust)** variant — trained with modality dropout so it works well with any subset of inputs. For maximum quality when all modalities are available, use `"asterisk-labs/betaearth-segformer-film"` instead.

In [ ]:
from betaearth import BetaEarth

model = BetaEarth.from_pretrained()  # default: robust variant
print(model)

## 3. Load a sample Sentinel-2 tile from Major TOM

We read a single row directly from a Major TOM parquet shard on HuggingFace — no full dataset download needed.

In [ ]:
import io
import numpy as np
import rasterio
from fsspec.parquet import open_parquet_file
import pyarrow.parquet as pq
from PIL import Image

PARQUET_FILE = "part_02530"
ROW_INDEX = 222

# Columns BetaEarth needs (9 bands: 4×10m + 5×20m) + metadata
band_names_10m = ["B02", "B03", "B04", "B08"]
band_names_20m = ["B05", "B06", "B07", "B11", "B12"]
all_columns = band_names_10m + band_names_20m + ["product_id", "grid_cell", "thumbnail"]

url = f"https://huggingface.co/datasets/Major-TOM/Core-S2L2A/resolve/main/images/{PARQUET_FILE}.parquet"
with open_parquet_file(url, columns=all_columns) as f:
    with pq.ParquetFile(f) as pf:
        tile = pf.read_row_group(ROW_INDEX, columns=all_columns)

print(f"Grid cell: {tile['grid_cell'][0].as_py()}")
print(f"Product: {tile['product_id'][0].as_py()}")

In [ ]:
def decode_band(band_bytes):
    """Decode a GeoTIFF band from in-memory bytes."""
    with rasterio.open(io.BytesIO(band_bytes)) as src:
        return src.read(1)

# Stack 9 bands in model order: [B02,B03,B04,B08,B05,B06,B07,B11,B12]
bands = []
for b in band_names_10m:
    bands.append(decode_band(tile[b][0].as_py()))

for b in band_names_20m:
    arr = decode_band(tile[b][0].as_py())  # 534x534
    arr = np.array(Image.fromarray(arr).resize((1068, 1068), Image.BILINEAR))
    bands.append(arr)

s2_l2a = np.stack(bands, axis=0).astype(np.uint16)  # (9, 1068, 1068)

# Extract day-of-year from product ID timestamp
product_id = tile["product_id"][0].as_py()
from datetime import datetime
ts = product_id.split("_")[2]  # e.g. "20190204T071109"
doy = datetime.strptime(ts[:8], "%Y%m%d").timetuple().tm_yday

print(f"S2 L2A: {s2_l2a.shape}, DOY: {doy}")

## 4. Predict embeddings

In [ ]:
embedding = model.predict(s2_l2a=s2_l2a, doy=doy)
print(f"Predicted embedding: {embedding.shape}, dtype={embedding.dtype}")
print(f"L2 norm at centre pixel: {np.linalg.norm(embedding[534, 534]):.4f}")

## 5. Visualise: PCA-RGB

Project the 64-dimensional embedding to 3 principal components for visualisation.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def pca_rgb(emb, pca_model=None):
    """Project (H, W, D) embedding to (H, W, 3) RGB via PCA."""
    H, W, D = emb.shape
    flat = emb.reshape(-1, D)
    if pca_model is None:
        pca_model = PCA(n_components=3)
        pca_model.fit(flat)
    rgb = pca_model.transform(flat).reshape(H, W, 3)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / max(hi - lo, 1e-8), 0, 1)
    return rgb, pca_model

# S2 true-colour for reference
s2_rgb = np.stack([s2_l2a[2], s2_l2a[1], s2_l2a[0]], axis=-1)  # B04, B03, B02
s2_rgb = np.clip(s2_rgb.astype(np.float32) / 3000, 0, 1)

pred_rgb, pca_model = pca_rgb(embedding)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(s2_rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[0].axis("off")
axes[1].imshow(pred_rgb)
axes[1].set_title("BetaEarth Embedding (PCA-RGB)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. Model variants

BetaEarth provides **8 model variants** with different trade-offs:

| Model | Cos Sim | Params | Key property |
|-------|---------|--------|-------|
| **`betaearth-segformer-film-robust`** | 0.873 | 104.8M | **Default — works with any input subset** |
| `betaearth-segformer-film` | **0.886** | 0.3M | Best all-modality quality |
| `betaearth-segformer-film-hilr` | 0.886 | 0.3M | Alternative frozen variant |
| `betaearth-segformer-film-scratch` | 0.883 | 104.8M | End-to-end trainable |
| `betaearth-segformer` | 0.880 | 104.8M | **No timestamp needed** |
| `betaearth-dinov3-vitl16` | 0.874 | 2.1M | DINOv3 satellite-pretrained |
| `betaearth-dinov3-vits16` | 0.861 | 1.8M | DINOv3 lightweight |
| `betaearth-rgb-only` | 0.836 | 26.3M | **Only 3 RGB bands needed** |

Below we compare the default robust model with two simpler alternatives.

In [ ]:
# RGB-only model — just 3 bands + day-of-year
model_rgb = BetaEarth.from_pretrained("asterisk-labs/betaearth-rgb-only")
s2_rgb_input = s2_l2a[[2, 1, 0]]  # B04, B03, B02 → (3, H, W)
emb_rgb = model_rgb.predict(s2_l1c=s2_rgb_input, doy=doy)

# No-FiLM model — no day-of-year needed
model_nofilm = BetaEarth.from_pretrained("asterisk-labs/betaearth-segformer")
emb_nofilm = model_nofilm.predict(s2_l2a=s2_l2a)  # no doy argument

print(f"Full model embedding:  {embedding.shape}")
print(f"RGB-only embedding:    {emb_rgb.shape}")
print(f"No-FiLM embedding:     {emb_nofilm.shape}")

In [ ]:
# PCA-RGB comparison — all three models, same PCA fitted on the full model
pred_rgb_nofilm, _ = pca_rgb(emb_nofilm, pca_model)
pred_rgb_rgbonly, _ = pca_rgb(emb_rgb, pca_model)

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
axes[0].imshow(s2_rgb)
axes[0].set_title("S2 RGB input")
axes[1].imshow(pred_rgb)
axes[1].set_title("Full model (0.886)")
axes[2].imshow(pred_rgb_nofilm)
axes[2].set_title("No FiLM (0.880)")
axes[3].imshow(pred_rgb_rgbonly)
axes[3].set_title("RGB-only (0.836)")
for ax in axes:
    ax.axis("off")
plt.suptitle("BetaEarth embeddings (PCA-RGB) — same PCA projection", fontsize=14)
plt.tight_layout()
plt.show()

## Next steps

- Add **Sentinel-1 and DEM** for higher fidelity: `model.predict(s2_l2a=..., s1=..., dem=..., doy=...)`
- **Average multiple timestamps** for annual embeddings (saturates at ~4 observations)
- The robust model handles **missing modalities** gracefully — L2A-only: 0.755, S1-only: 0.712
- Browse all 8 models at [huggingface.co/asterisk-labs](https://huggingface.co/asterisk-labs)
- Code and paper: [github.com/asterisk-labs/beta-earth](https://github.com/asterisk-labs/beta-earth)